In [1]:
import hippynn
import numpy as np
import matplotlib.pyplot as plt
from hippynn.pretraining import calculate_min_dists

In [2]:
import torch

In [3]:
from hippynn.databases import AseDatabase

In [4]:
import torch
torch.set_default_dtype(torch.float64)

In [ ]:
database = AseDatabase('./','compiled.json',inputs=['Z'],targets=[],allow_unfound=True,seed=0)

In [ ]:
database.split_the_rest('all')

In [ ]:
database.write_npz('compiled.npz',record_split_masks=False)

In [9]:
xtb_single_atom_ref_es = {'H': -10.707211383396714, 
 'He': -47.432891698445495,
 'Li': -4.900000175455953,
 'Be': -15.486162554518229,
 'B': -25.917120371751917,
 'C': -48.847445262804705,
 'N': -71.00681805517411,
 'O': -102.57117256025786,
 'F': -125.69864294466228,
 'Ne': -161.42379378015602,
 'Na': -4.5469341628136,
 'Mg': -12.67981645403045,
 'Al': -24.63524632585141,
 'Si': -42.76062738849161,
 'P': -64.70342656532247,
 'S': -85.66881795502964,
 'Cl': -121.97572181135438,
 'Ar': -116.4386981693596,
 'K': -4.510348161503552,
 'Ca': -10.113012362120033,
 'Sc': -23.243511328068923,
 'Ti': -37.19952407328606,
 'V': -46.75256780519609,
 'Cr': -47.551829431041725,
 'Mn': -71.27032275557823,
 'Fe': -80.14401872919589,
 'Co': -95.42461229338535,
 'Ni': -127.03165993183204,
 'Cu': -101.98844165193441,
 'Zn': -14.354588513999577,
 'Ga': -29.418551497128806,
 'Ge': -49.249990531246375,
 'As': -60.93788396017277,
 'Se': -84.9113939279083,
 'Br': -110.16092538829801,
 'Kr': -116.24311016235609,
 'Rb': -4.353793155897734,
 'Sr': -12.583384450577476,
 'Y': -32.513603426171194,
 'Zr': -35.664705344038914,
 'Nb': -48.468943633382366,
 'Mo': -47.103307394758325,
 'Tc': -67.47438866812136,
 'Ru': -77.53081994875949,
 'Rh': -106.01053867238016,
 'Pd': -119.99800276414216,
 'Ag': -103.99479372376777,
 'Cd': -14.504682519374043,
 'In': -30.63832754080574,
 'Sn': -54.773706848758266,
 'Sb': -58.891664996211276,
 'Te': -81.88153581941742,
 'I': -102.84897812647684,
 'Xe': -105.67782578404142,
 'Cs': -4.04170614472273,
 'Ba': -11.800000422526582,
 'La': -32.78408804872581,
 'Ce': -24.476883526857367,
 'Pr': -24.30981661010049,
 'Nd': -24.142748696083178,
 'Pm': -23.975681784199946,
 'Sm': -23.80861387345064,
 'Eu': -23.641546957707547,
 'Gd': -23.474481043951293,
 'Tb': -23.30741258426,
 'Dy': -23.140346671193583,
 'Ho': -22.973279758167216,
 'Er': -22.80621184515164,
 'Tm': -22.63914493213882,
 'Yb': -22.472076019126465,
 'Lu': -22.305010106114292,
 'Hf': -35.70130487846373,
 'Ta': -51.83261682874072,
 'W': -60.25302489724785,
 'Re': -81.80817630252322,
 'Os': -81.31693407962229,
 'Ir': -17.446427065227073,
 'Pt': -120.7493981742078,
 'Au': -103.47454570514782,
 'Hg': -23.076132826294845,
 'Tl': -39.14861684553654,
 'Pb': -59.99677903578255,
 'Bi': -61.67878109601047,
 'Po': -74.41720155213928,
 'At': -81.64893636735967,
 'Rn': -104.97843975899828}

In [10]:
import ase

In [11]:
from hippynn.databases import NPZDatabase

In [10]:
dbs = database.splits['all']
forces = dbs['forces']

In [11]:
species = dbs['numbers']

In [12]:
num_en_map = {ase.data.atomic_numbers[k]:v for k,v in xtb_single_atom_ref_es.items()}
num_en_map[0] = 0
get_energy_from_numbers = np.vectorize(num_en_map.__getitem__)
atom_en = get_energy_from_numbers(species)
sys_en = atom_en.sum(axis=-1)

In [ ]:
plt.scatter(sys_en,dbs['energy'])
plt.xlabel("atomic energy")
plt.ylabel("total energy")
plt.show()

In [ ]:
real_forces=forces[species!=0]

In [ ]:
new_db=NPZDatabase('./compiled.npz',inputs=[],targets=[],allow_unfound=True,seed=0)

species = new_db.arr_dict['numbers']
num_en_map = {ase.data.atomic_numbers[k]:v for k,v in xtb_single_atom_ref_es.items()}
num_en_map[0] = 0
get_energy_from_numbers = np.vectorize(num_en_map.__getitem__)
atom_en = get_energy_from_numbers(species)
sys_en = atom_en.sum(axis=-1)
energy_minus_self=new_db.arr_dict['energy']-sys_en

new_db.arr_dict['total_energy']=new_db.arr_dict['energy']
new_db.arr_dict['energy']=energy_minus_self
del new_db.arr_dict['energy_per_atom']

In [ ]:
plt.hist(new_db.arr_dict['energy'],bins=1000)
plt.yscale('log')
plt.xlabel("residual energy")
plt.ylabel("count")
plt.show()

In [ ]:
from hippynn.pretraining import calculate_max_system_force
max_forces =calculate_max_system_force(new_db.arr_dict,species_name='numbers',force_name='forces')
plt.hist(max_forces.flatten(),bins=1000)
plt.yscale('log')
plt.xlabel("residual energy")
plt.ylabel("count")
plt.show()

In [ ]:
new_db.remove_high_property('energy',atomwise=False, std_factor=4,norm_per_atom=True,species_key='numbers')
# new_db.remove_high_property('forces',atomwise=True, std_factor=4, species_key='numbers',norm_axis=-1)

In [ ]:
from hippynn.pretraining import calculate_max_system_force
max_forces =calculate_max_system_force(new_db.arr_dict,species_name='numbers',force_name='forces')
force_thresh=100 # this is big 

# mask = max_forces.flatten()>force_thresh
# new_db.make_explicit_split_bool('max_force_cut',mask.numpy())

In [ ]:
plt.hist(max_forces.flatten(),bins=100)
plt.yscale('log')
plt.show()

In [ ]:
min_d_thresh=0.5
min_d = calculate_min_dists(new_db.arr_dict,species_name='numbers',positions_name='positions',dist_hard_max=5.)

mask = min_d<min_d_thresh
new_db.make_explicit_split_bool('minimum_distance_cut',mask.numpy())

In [ ]:
plt.hist(min_d.flatten(),bins=100)
plt.yscale('log')
plt.show()

In [ ]:
n_atoms = (new_db.arr_dict['numbers']!=0).sum(axis=-1).flatten()
en_per_atom = new_db.arr_dict['energy'].flatten()/n_atoms
new_db.arr_dict['energy_per_atom']= en_per_atom
plt.hist(en_per_atom,bins=1000)
plt.yscale('log')
plt.xlabel("energy per atom")
plt.ylabel("count")
plt.show()

In [ ]:
new_forces = new_db.arr_dict['forces'][new_db.arr_dict['numbers']!=0]
new_force_mag = np.linalg.norm(new_forces,axis=1)
plt.hist(new_force_mag,bins=100)
plt.yscale('log')
plt.show()

In [ ]:
plt.hist(min_d,bins=100)
plt.yscale('log')
plt.show()

In [ ]:
new_db.split_the_rest('all')
for k in list(new_db.splits.keys()):
    if k == 'all':
        continue
    else:
        print("removing split",k)
        del new_db.splits[k]
new_db.write_npz('trimmed.npz',record_split_masks=False,overwrite=True)

In [ ]:
help(new_db.write_npz)

In [ ]:
help(new_db.remove_high_property)